# Actividad en semana 2

In [8]:
from faker import Faker
import random
from datetime import datetime

fake = Faker('es_MX')

def generar_datos_ejemplo(n=100):
    datos = [ ]
    for _ in range(n):
        registro = {
            "nombre": fake.name(),
            "ciudad": fake.city(),
            "fecha_reporte": fake.date_between(start_date='-2y', end_date='today').isoformat(),
            "categoria": random.choice(["Seguridad","Salud","Infraestructura","Servicios"]),
            "Descripcion": fake.sentence(nb_words = 10),
            "Gravedad": random.randint(1,9),
            "Latitud": float(fake.latitude()),
            "Longitud": float(fake.longitude())
        }
        datos.append(registro)
    return datos

mis_datos = generar_datos_ejemplo(10)
print(f"Se generaron {len(mis_datos)} registros de ejemplo")

Se generaron 10 registros de ejemplo


In [9]:
import pandas as pd

df = pd.DataFrame(mis_datos)
display(df)

,nombre,ciudad,fecha_reporte,categoria,Descripcion,Gravedad,Latitud,Longitud
0,Ing. Gloria Sandoval,San Cristal los altos,2025-02-25,Seguridad,Cama quizá nuevas consejo temas relaciones aqu...,9,30.028779,58.657556
1,Julio César José Manuél Matías Barragán,Vieja Myanmar,2025-04-07,Infraestructura,Nuevas orden porque capital d quizá rodríguez ...,2,-81.475911,132.623240
2,Adán Aldo Soria,San Miguel Ángel los bajos,2024-12-07,Servicios,Sus considera francia través otras mano lo.,1,-79.977019,92.043029
3,Lic. Itzel Carmona,San Claudia los bajos,2024-06-07,Servicios,Mes resto ha empresa llegado pronto.,8,8.918750,-123.366368
4,Eloy Genaro Villa,San Aurelio los altos,2026-03-01,Salud,América pese manuel baja país unos éxito.,8,26.488979,-179.291993
5,Iván Diego Ornelas Gaitán,San Ivonne los altos,2026-02-04,Seguridad,Misma equipo garcía según si partes cultura.,4,2.455789,-38.355824
6,Sr. José Manuél Vallejo,Nueva República Federal Democrática de Nepal,2025-02-09,Servicios,Naturaleza congreso problema francisco eleccio...,4,-48.748194,66.303026
7,Rafaél Hernández Barraza,Vieja Sudán,2024-10-12,Salud,Social popular los las puesto formación e cabe...,1,-46.948087,41.675244
8,David Raya Acuña,Nueva Tayikistán,2026-03-07,Infraestructura,Director industria izquierda podía santa le cu...,4,34.585388,-9.267458
9,Humberto Amaya,San Rebeca de la Montaña,2025-11-20,Infraestructura,Muchos tarde cuales p pese finalmente hablar t...,3,25.739276,-134.030673


In [13]:
import mysql.connector

def insertar_en_mysql(datos):
    try:
        conexion = mysql.connector.connect(
            host="localhost",
            user="root",
            database="exccomp2"
        )
        cursor = conexion.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS reportes (    
            id INT AUTO_INCREMENT PRIMARY KEY,
            nombre VARCHAR(100),
            ciudad VARCHAR(50),
            fecha_reporte DATE,
            categoria VARCHAR(50),
            descripcion TEXT,
            gravedad INT
        )
        """)

        query = "INSERT INTO reportes(nombre, ciudad, fecha_reporte, categoria, descripcion, gravedad) VALUES (%s, %s, %s, %s, %s, %s)"
        valores = [
            (
            d.get("nombre", "Sin nombre"),
                d.get("ciudad", "Desconocida"),
                d.get("fecha_reporte"), 
                d.get("categoria", "General"),
                d.get("descripcion", "Sin descripción"), 
                d.get("gravedad", 0)
            ) for d in datos
        ]

        cursor.executemany(query, valores)
        conexion.commit()
        print(f"Exito: {cursor.rowcount} registros insertados en MySQL.")
       
    except mysql.connector.Error as err:
        print(f"Error: {err}")
    finally:
        if conexion.is_connected():
            cursor.close()
            conexion.close()
 
insertar_en_mysql(mis_datos)

Exito: 10 registros insertados en MySQL.


In [ ]:
from pymongo import MongoClient

def insertar_mongodb(datos):
    try:
        cliente = MongoClient("mongodb://localhost:27017/")
        db = cliente["clase_datos"]
        coleccion = db["reportes"]

        resultado = coleccion.insert_many(datos)
        print("Datos insertados")
    except Exception as e:
        print(f"Error {e}")

insertar_mongodb(mis_datos)

In [ ]:
import mysql.connector
import pandas as pd

def leer_sql():
    try:
        conexion = mysql.connector.connect(
            host="localhost",
            user="root",
            database="exccomp2"
        )

        query = "SELECT*FROM reportes"
        df_mysql = pd.read_sql(query, conexion)

        print("Dtaos leidos chidos!")
        return df_mysql
    
    except mysql.connector.Error as err:
        print(f"Error: (err)")
    finally:
        if conexion.is_connected():
            conexion.close()

dataFrame_sql = leer_sql()
display(dataFrame_sql)